# 02 — Quantum simulation performance and memory

**Level:** intermediate. **Environment:** Qiskit Aer CPU; compare the same experiment on an approved GPU using Lab 01 and the immutable Aer-GPU image. **Time:** seconds locally, queue-dependent remotely. **Result:** memory estimates and CPU reference timings.

Double-precision pure statevector amplitude storage requires `16 * 2**n` bytes before workspace, metadata, CUDA buffers, multi-device replication or additional states. Do not request a large H200 allocation simply because one exists.

In [ ]:
from math import log2
def statevector_gib(n, bytes_per_complex=16):
    return bytes_per_complex * (2 ** n) / (1024 ** 3)
for n in [20, 25, 30, 32, 34]:
    print(f"{n:2d} qubits: {statevector_gib(n):9.3f} GiB raw complex128 amplitudes")
assert abs(statevector_gib(30) - 16.0) < 1e-9

## Controlled CPU baseline

For a tiny reference circuit, compare **numerical correctness** and measured wall time; never claim the GPU will be faster at this scale. Record shots, seed, method, precision, hardware and image digest for a legitimate benchmark.

In [ ]:
import time
from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator

def ghz(n):
    q = QuantumCircuit(n, n)
    q.h(0)
    for i in range(1, n):
        q.cx(i-1, i)
    q.measure(range(n), range(n))
    return q

cpu = AerSimulator(method="statevector", device="CPU")
for n in (10, 14):
    circ = transpile(ghz(n), cpu)
    start = time.perf_counter()
    res = cpu.run(circ, shots=1024, seed_simulator=42).result()
    elapsed = time.perf_counter() - start
    observed = res.get_counts()
    assert res.success and set(observed) <= {"0"*n, "1"*n}
    print({"n":n, "seconds":round(elapsed,4), "shots":sum(observed.values()), "device":"CPU"})

## GPU exercise

Submit `slurm/aer-gpu.sbatch` through notebook 01 with `AER_QUBITS` set to an **approved**, memory-justified size. Compare its run metadata and actual `sacct` TRES accounting. The local CPU code does not reserve a GPU. GPU validation and benchmarking require observed GPU device identification and matched computational settings.